# TrkA 필터 교정 · 실험 라벨 대비 점수 분포 · v0.1.1

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_protein/blob/main/notebooks/02_calibrate_filter_T4.ipynb?v=0.1.1)

T4에서 전체 실행하면 **실험 결합 라벨이 붙은 공개 설계**를 교정된 AF2 initial guess 필터로 채점합니다. 결합체 6개 전부와 비결합체 14개를 뽑아 AF2를 20회 실행하며, 약 40분이 걸립니다.

TrkA_88 재현([진단 노트북](https://colab.research.google.com/github/HisameOgasahara/tmp_protein/blob/main/diagnostics/90_af2_conditions_T4.ipynb?v=0.2.0))은 파이프라인이 정상임을 확인한 것입니다. 이 노트북은 다른 것을 봅니다 — **그 점수가 실제 결합 여부를 가르는가.**

| 그룹 | 개수 | 출처 |
| --- | --- | --- |
| 실험에서 결합 확인 | 6 (전수) | `TrkA_designs.csv`의 `Binder=True` |
| 실험에서 결합 안 됨 | 14 (시드 0으로 추출) | 나머지 89개에서 |

결합체는 이 표적에 존재하는 것이 6개 전부라 표본이 아니라 전수입니다.

공개 자료 아카이브(`RFdiffusion_designs.zip`, 251.7 MB)를 Colab에서 받아 TrkA 부분만 풉니다. 설계별로 표적 사이드체인을 1WWW 체인 X에서 이식하고 공개 서열을 입힌 뒤, 준비된 20개를 업스트림 `predict.py`에 **한 번에** 넘깁니다. 모델을 1회만 적재하려는 것입니다.

결합여 길이는 설계마다 다릅니다(65~94잔기). 표적 101잔기는 모든 설계에서 동일합니다.

통과 기준은 `pae_interaction < 10`, `plddt_binder > 80`, `binder_aligned_rmsd < 2 Å`입니다.

**예상되는 결과를 미리 적어둡니다.** 95개는 저자들이 이미 계산 필터를 통과시켜 실험까지 보낸 집합이므로 두 그룹의 분포가 크게 겹칠 수 있습니다. 그 경우의 결론은 "이 지표는 후보를 줄이는 체이고 통과 후 실제 결합률은 6/95 ≈ 6.3%"가 됩니다. 분포가 갈라지면 그것도 결론입니다. 어느 쪽이든 설계 단계의 점수를 해석할 기준이 생깁니다.

[공개 자료](https://doi.org/10.6084/m9.figshare.22662094.v1) · [원본 AF2 코드](https://github.com/nrbennet/dl_binder_design/blob/cafa3853ac94dceb1b908c8d9e6954d71749871a/af2_initial_guess/predict.py)


In [ ]:
from pathlib import Path
from tempfile import mkdtemp
import subprocess
import sys

checkout = Path(mkdtemp(prefix="trka88_main_", dir="/content")) / "repo"
clone = subprocess.run(
    ["git", "clone", "--depth", "1", "--single-branch", "--branch", "main",
     "https://github.com/HisameOgasahara/tmp_protein.git", str(checkout)],
    capture_output=True, text=True,
)
if clone.returncode:
    print(clone.stderr)
    clone.check_returncode()
revision = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
print(f"main: {revision[:12]}")


In [ ]:
import json
import pandas as pd
from IPython.display import display

log = checkout.parent / "trka88_calibration.log"
result_zip = None
with log.open("w", encoding="utf-8") as handle:
    with subprocess.Popen(
        [sys.executable, "-u", str(checkout / "src/af2_filter/calibration_workflow.py"), revision],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8",
    ) as process:
        for line in process.stdout:
            handle.write(line)
            handle.flush()
            print(line, end="", flush=True)
            if line.strip().endswith(".zip"):
                result_zip = Path(line.strip())
        if process.wait():
            raise RuntimeError(f"Calibration failed. Full log: {log}")
if result_zip is None or not result_zip.is_file():
    raise FileNotFoundError(f"Missing result ZIP. Full log: {log}")

folder = result_zip.with_suffix("")
table = folder / "calibration.csv"
if not table.is_file():
    print("설치 또는 준비 단계 오류 기록이 ZIP에 저장됐습니다.")
else:
    frame = pd.read_csv(table)
    frame["group"] = frame["binder_experimental"].map({True: "결합", False: "비결합"})
    columns = ["design", "group", "passes_filter", "pae_interaction", "plddt_binder",
               "binder_aligned_rmsd", "rmsd_vs_design", "rmsd_vs_published_af2", "length"]
    print()
    print("=== 설계별 점수 (결합면 PAE 순) ===")
    display(frame.reindex(columns=columns).sort_values("pae_interaction").round(3))
    print()
    print("=== 그룹별 분포 ===")
    stats = frame.groupby("group")["pae_interaction"].agg(["count", "min", "median", "max"])
    stats["필터 통과"] = frame.groupby("group")["passes_filter"].sum()
    display(stats.round(3))
    summary = folder / "calibration_summary.json"
    if summary.is_file():
        print(json.dumps(json.loads(summary.read_text()), indent=2, ensure_ascii=False))
print(result_zip)


In [ ]:
from google.colab import files
files.download(str(result_zip))
